# 02 - Engenharia de Dados

Este notebook realiza a primeira camada de validação, qualidade e enriquecimento dos dados do projeto **Agro Leads Orchestrator**.

Nesta etapa, serão avaliados:

- estrutura da base;
- quantidade de registros;
- valores nulos;
- duplicidades;
- distribuição de status;
- distribuição agrícola por cultura e estágio;
- estatísticas descritivas do score;
- detecção de outliers;
- criação de features operacionais.

## Objetivo

O objetivo deste notebook é transformar a base sintética em uma base analítica mais confiável, validada e preparada para as próximas etapas do projeto.

Esta etapa representa uma rotina típica de Engenharia de Dados e Ciência de Dados antes da construção de modelos, simuladores ou dashboards.

In [1]:
from pathlib import Path
import sqlite3
import sys

import pandas as pd
import numpy as np

In [2]:
#Busca raiz do projeto

RAIZ_PROJETO = Path.cwd()

if RAIZ_PROJETO.name == "notebooks":
    RAIZ_PROJETO = RAIZ_PROJETO.parent

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.append(str(RAIZ_PROJETO))

print("Raiz do projeto:", RAIZ_PROJETO)

Raiz do projeto: c:\Users\DataCore\Desktop\projetos_Git\agro-leads-orchestrator


In [3]:
#Importar funções internas

from src.estatisticas import (
    calcular_percentual_nulos,
    calcular_duplicidades,
    calcular_estatisticas_score,
    detectar_outliers_iqr,
    criar_faixa_score,
    criar_indicadores_operacionais
)

In [4]:
#Definir caminho do banco

CAMINHO_BANCO = RAIZ_PROJETO / "dados" / "agro_leads.db"

print("Banco localizado em:", CAMINHO_BANCO)
print("Banco existe?", CAMINHO_BANCO.exists())

Banco localizado em: c:\Users\DataCore\Desktop\projetos_Git\agro-leads-orchestrator\dados\agro_leads.db
Banco existe? True


In [5]:
#Criar conexão com SQLite

conexao = sqlite3.connect(CAMINHO_BANCO)

print("Conexão criada com sucesso.")

Conexão criada com sucesso.


In [6]:
#Listar tabelas disponíveis

consulta_tabelas = """
SELECT name AS tabela
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

pd.read_sql_query(consulta_tabelas, conexao)

,tabela
0,eventos_contato
1,leads
2,sqlite_sequence
3,sqlite_stat1


In [7]:
#Carregar base leads

consulta_leads = """
SELECT
    id_cliente,
    nome,
    telefone,
    cultura,
    estagio_atual,
    status_atual,
    ultimo_contato,
    cooldown_ate,
    score_prioridade
FROM leads;
"""

dados_leads = pd.read_sql_query(consulta_leads, conexao)

dados_leads.head()

,id_cliente,nome,telefone,cultura,estagio_atual,status_atual,ultimo_contato,cooldown_ate,score_prioridade
0,1,Lucas Martins,5512900000001,Soja,Desenvolvimento,Disponível,NaN,NaN,71.86
1,2,Fernando Lima,5517900000002,Soja,Plantio,Disponível,2026-06-03 19:08:57,NaN,73.28
2,3,Marcelo Gomes,5511900000003,Milho,Safra,Disponível,2026-06-09 05:00:57,NaN,80.97
3,4,Daniel Araújo,5535900000004,Milho,Plantio,Disponível,NaN,NaN,25.93
4,5,João Silva,5543900000005,Cana,Safra,Convertido,2026-06-23 04:09:57,2026-07-28 17:02:57,0.00


In [8]:
#Dimensão da base

quantidade_linhas, quantidade_colunas = dados_leads.shape

print(f"Total de linhas: {quantidade_linhas:,}")
print(f"Total de colunas: {quantidade_colunas}")

Total de linhas: 950,000
Total de colunas: 9


In [9]:
dados_leads.info()

<class 'pandas.DataFrame'>
RangeIndex: 950000 entries, 0 to 949999
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id_cliente        950000 non-null  int64  
 1   nome              950000 non-null  str    
 2   telefone          950000 non-null  str    
 3   cultura           950000 non-null  str    
 4   estagio_atual     950000 non-null  str    
 5   status_atual      950000 non-null  str    
 6   ultimo_contato    779101 non-null  str    
 7   cooldown_ate      170960 non-null  str    
 8   score_prioridade  950000 non-null  float64
dtypes: float64(1), int64(1), str(7)
memory usage: 65.2 MB


In [10]:
#Converter colunas de data

dados_leads["ultimo_contato"] = pd.to_datetime(
    dados_leads["ultimo_contato"],
    errors="coerce"
)

dados_leads["cooldown_ate"] = pd.to_datetime(
    dados_leads["cooldown_ate"],
    errors="coerce"
)

dados_leads.dtypes

id_cliente                   int64
nome                           str
telefone                       str
cultura                        str
estagio_atual                  str
status_atual                   str
ultimo_contato      datetime64[us]
cooldown_ate        datetime64[us]
score_prioridade           float64
dtype: object

In [ ]:
#Avaliar valores nulos
#é esperado existir nulo em ultimo_contato e cooldown_ate, 
#porque nem todo lead teve contato anterior ou cooldown.

resumo_nulos = calcular_percentual_nulos(dados_leads)

resumo_nulos

,coluna,quantidade_nulos,percentual_nulos
7,cooldown_ate,779040,82.00
6,ultimo_contato,170899,17.99
0,id_cliente,0,0.00
1,nome,0,0.00
2,telefone,0,0.00
4,estagio_atual,0,0.00
3,cultura,0,0.00
5,status_atual,0,0.00
8,score_prioridade,0,0.00


In [13]:
#Avaliar duplicidade em id_clientes

duplicidade_id = calcular_duplicidades(
    dados=dados_leads,
    coluna="id_cliente"
)

duplicidade_id

{'coluna': 'id_cliente',
 'total_registros': 950000,
 'total_unicos': 950000,
 'total_duplicados': 0,
 'percentual_duplicados': 0.0}

In [15]:
#Avaliar duplicidade na coluna telefone

duplicidade_telefone = calcular_duplicidades(
    dados=dados_leads,
    coluna="telefone"
)

duplicidade_telefone

{'coluna': 'telefone',
 'total_registros': 950000,
 'total_unicos': 950000,
 'total_duplicados': 0,
 'percentual_duplicados': 0.0}

In [16]:
#Distribuição por status

distribuicao_status = (
    dados_leads
    .groupby("status_atual")
    .agg(
        quantidade=("id_cliente", "count"),
        score_medio=("score_prioridade", "mean")
    )
    .reset_index()
)

distribuicao_status["percentual"] = (
    distribuicao_status["quantidade"]
    / distribuicao_status["quantidade"].sum()
    * 100
)

distribuicao_status = distribuicao_status.sort_values(
    by="quantidade",
    ascending=False
)

distribuicao_status.round(2)

,status_atual,quantidade,score_medio,percentual
1,Disponível,683811,59.64,71.98
3,Em Cooldown,114146,8.95,12.02
4,Fila Prioritária,57024,107.59,6.00
0,Convertido,56814,0.00,5.98
2,Em Atendimento,38205,17.89,4.02


In [18]:
#Distribuição por cultura

distribuicao_cultura = (
    dados_leads
    .groupby("cultura")
    .agg(
        quantidade=("id_cliente", "count"),
        score_medio=("score_prioridade", "mean"),
        score_minimo=("score_prioridade", "min"),
        score_maximo=("score_prioridade", "max")
    )
    .reset_index()
    .sort_values(by="quantidade", ascending=False)
)

distribuicao_cultura.round(2)

,cultura,quantidade,score_medio,score_minimo,score_maximo
0,Cana,428265,54.46,0.0,250.0
2,Soja,331745,49.36,0.0,250.0
1,Milho,189990,46.97,0.0,250.0


In [19]:
#Distribuição por estágio agrícola

distribuicao_estagio = (
    dados_leads
    .groupby("estagio_atual")
    .agg(
        quantidade=("id_cliente", "count"),
        score_medio=("score_prioridade", "mean"),
        score_mediano=("score_prioridade", "median")
    )
    .reset_index()
    .sort_values(by="score_medio", ascending=False)
)

distribuicao_estagio.round(2)

,estagio_atual,quantidade,score_medio,score_mediano
2,Plantio,237277,66.69,70.48
3,Safra,237541,59.89,63.39
0,Desenvolvimento,284857,44.42,46.97
1,Entresafra,190325,31.09,32.88


In [20]:
#Cultura X estágio agrícola

analise_cultura_estagio = (
    dados_leads
    .groupby(["cultura", "estagio_atual"])
    .agg(
        quantidade=("id_cliente", "count"),
        score_medio=("score_prioridade", "mean"),
        score_mediano=("score_prioridade", "median")
    )
    .reset_index()
    .sort_values(by="score_medio", ascending=False)
)

analise_cultura_estagio.round(2)

,cultura,estagio_atual,quantidade,score_medio,score_mediano
2,Cana,Plantio,107413,71.03,75.39
10,Soja,Plantio,82801,64.19,68.28
3,Cana,Safra,106927,63.73,67.74
6,Milho,Plantio,47063,61.14,64.98
11,Soja,Safra,82859,57.81,61.60
7,Milho,Safra,47755,54.91,58.35
0,Cana,Desenvolvimento,128022,47.23,50.21
8,Soja,Desenvolvimento,99708,42.86,45.54
4,Milho,Desenvolvimento,57127,40.85,43.37
1,Cana,Entresafra,85903,32.97,35.07


In [21]:
#Estatística descritiva do score

estatisticas_score = calcular_estatisticas_score(dados_leads)

estatisticas_score

,score_prioridade
count,950000.00
mean,51.18
std,34.64
min,0.00
25%,24.96
50%,49.52
75%,72.74
90%,94.53
95%,109.08
max,250.00


In [22]:
#Desvio padrão por status

desvio_score_status = (
    dados_leads
    .groupby("status_atual")
    .agg(
        quantidade=("id_cliente", "count"),
        media_score=("score_prioridade", "mean"),
        desvio_padrao_score=("score_prioridade", "std"),
        score_minimo=("score_prioridade", "min"),
        score_maximo=("score_prioridade", "max")
    )
    .reset_index()
    .sort_values(by="media_score", ascending=False)
)

desvio_score_status.round(2)

,status_atual,quantidade,media_score,desvio_padrao_score,score_minimo,score_maximo
4,Fila Prioritária,57024,107.59,43.59,1.26,250.00
1,Disponível,683811,59.64,24.28,0.66,165.00
2,Em Atendimento,38205,17.89,7.30,0.23,48.14
3,Em Cooldown,114146,8.95,3.64,0.10,24.75
0,Convertido,56814,0.00,0.00,0.00,0.00


In [23]:
#Quartis por estágio agricola

quartis_por_estagio = (
    dados_leads
    .groupby("estagio_atual")["score_prioridade"]
    .quantile([0.25, 0.50, 0.75, 0.90])
    .unstack()
    .reset_index()
)

quartis_por_estagio.columns = [
    "estagio_atual",
    "q1_25%",
    "mediana_50%",
    "q3_75%",
    "p90_90%"
]

quartis_por_estagio.round(2)

,estagio_atual,q1_25%,mediana_50%,q3_75%,p90_90%
0,Desenvolvimento,24.82,46.97,61.08,74.49
1,Entresafra,17.33,32.88,42.76,52.19
2,Plantio,37.51,70.48,91.65,111.93
3,Safra,33.39,63.39,82.42,100.38


In [24]:
#Detectar outliers de score

outliers_score = detectar_outliers_iqr(
    dados=dados_leads,
    coluna="score_prioridade"
)

print(f"Quantidade de outliers encontrados: {len(outliers_score):,}")

outliers_score.head()

Quantidade de outliers encontrados: 12,263


,id_cliente,nome,telefone,cultura,estagio_atual,status_atual,ultimo_contato,cooldown_ate,score_prioridade,limite_inferior,limite_superior
94,95,Roberto Lima,5543900000095,Soja,Plantio,Fila Prioritária,2026-06-28 01:12:57,NaT,182.49,-46.71,144.41
266,267,Luiz Barbosa,5518900000267,Cana,Safra,Fila Prioritária,2026-06-17 16:21:57,NaT,187.54,-46.71,144.41
424,425,Paulo Ribeiro,5567900000425,Soja,Safra,Fila Prioritária,NaT,NaT,151.96,-46.71,144.41
446,447,Daniel Costa,5535900000447,Soja,Plantio,Fila Prioritária,2026-06-08 17:10:57,NaT,199.40,-46.71,144.41
493,494,Eduardo Ribeiro,5512900000494,Cana,Plantio,Disponível,NaT,NaT,144.55,-46.71,144.41


In [25]:
#Percentual de outliers

percentual_outliers = len(outliers_score) / len(dados_leads) * 100

print(f"Percentual de outliers de score: {percentual_outliers:.2f}%")

Percentual de outliers de score: 1.29%


In [26]:
#Criar faixa de score

dados_leads_enriquecidos = criar_faixa_score(dados_leads)

dados_leads_enriquecidos[[
    "id_cliente",
    "cultura",
    "estagio_atual",
    "status_atual",
    "score_prioridade",
    "faixa_score"
]].head()

,id_cliente,cultura,estagio_atual,status_atual,score_prioridade,faixa_score
0,1,Soja,Desenvolvimento,Disponível,71.86,Médio
1,2,Soja,Plantio,Disponível,73.28,Médio
2,3,Milho,Safra,Disponível,80.97,Médio
3,4,Milho,Plantio,Disponível,25.93,Muito Baixo
4,5,Cana,Safra,Convertido,0.00,Muito Baixo


In [27]:
#Criar indicadores operacionais

dados_leads_enriquecidos = criar_indicadores_operacionais(
    dados_leads_enriquecidos
)

dados_leads_enriquecidos[[
    "id_cliente",
    "estagio_atual",
    "status_atual",
    "lead_em_momento_agricola_critico",
    "lead_disponivel_para_contato",
    "lead_engajado_whatsapp",
    "lead_bloqueado"
]].head()

,id_cliente,estagio_atual,status_atual,lead_em_momento_agricola_critico,lead_disponivel_para_contato,lead_engajado_whatsapp,lead_bloqueado
0,1,Desenvolvimento,Disponível,False,True,False,False
1,2,Plantio,Disponível,True,True,False,False
2,3,Safra,Disponível,True,True,False,False
3,4,Plantio,Disponível,True,True,False,False
4,5,Safra,Convertido,True,False,False,True


In [28]:
#Analisar faixa de score

analise_faixa_score = (
    dados_leads_enriquecidos
    .groupby("faixa_score", observed=True)
    .agg(
        quantidade=("id_cliente", "count"),
        score_medio=("score_prioridade", "mean")
    )
    .reset_index()
)

analise_faixa_score["percentual"] = (
    analise_faixa_score["quantidade"]
    / analise_faixa_score["quantidade"].sum()
    * 100
)

analise_faixa_score.round(2)

,faixa_score,quantidade,score_medio,percentual
0,Muito Baixo,272628,11.51,28.70
1,Baixo,317505,45.52,33.42
2,Médio,243171,73.35,25.60
3,Alto,96845,103.76,10.19
4,Prioridade Máxima,19851,158.28,2.09


In [29]:
#Leads críticos para operação comercial

leads_criticos = dados_leads_enriquecidos[
    (dados_leads_enriquecidos["lead_em_momento_agricola_critico"])
    & (dados_leads_enriquecidos["lead_disponivel_para_contato"])
].copy()

leads_criticos = leads_criticos.sort_values(
    by="score_prioridade",
    ascending=False
)

leads_criticos.head(20)

,id_cliente,nome,telefone,cultura,estagio_atual,status_atual,ultimo_contato,cooldown_ate,score_prioridade,faixa_score,lead_em_momento_agricola_critico,lead_disponivel_para_contato,lead_engajado_whatsapp,lead_bloqueado
85792,85793,Marcos Santos,5518900085793,Cana,Plantio,Disponível,2026-06-04 17:08:57,NaT,165.0,Prioridade Máxima,True,True,False,False
835792,835793,Henrique Lima,5564900835793,Cana,Plantio,Disponível,2026-06-23 18:49:09,NaT,165.0,Prioridade Máxima,True,True,False,False
11675,11676,Mateus Souza,5514900011676,Cana,Plantio,Disponível,2026-05-31 20:07:57,NaT,165.0,Prioridade Máxima,True,True,False,False
794574,794575,João Fernandes,5514900794575,Cana,Plantio,Disponível,2026-06-06 23:18:07,NaT,165.0,Prioridade Máxima,True,True,False,False
886516,886517,Carlos Barbosa,5517900886517,Cana,Plantio,Disponível,2026-06-05 07:07:09,NaT,165.0,Prioridade Máxima,True,True,False,False
260362,260363,José Rodrigues,5512900260363,Cana,Plantio,Disponível,2026-06-23 03:11:00,NaT,165.0,Prioridade Máxima,True,True,False,False
350300,350301,José Silva,5516900350301,Cana,Plantio,Disponível,NaT,NaT,165.0,Prioridade Máxima,True,True,False,False
121581,121582,Paulo Silva,5544900121582,Cana,Plantio,Disponível,2026-06-08 02:33:58,NaT,165.0,Prioridade Máxima,True,True,False,False
237712,237713,Rafael Fernandes,5544900237713,Cana,Plantio,Disponível,2026-06-02 09:46:00,NaT,165.0,Prioridade Máxima,True,True,False,False
147143,147144,Rafael Santos,5515900147144,Cana,Plantio,Disponível,2026-06-22 10:33:58,NaT,165.0,Prioridade Máxima,True,True,False,False


In [30]:
#Resumo

resumo_engenharia = {
    "total_leads": len(dados_leads_enriquecidos),
    "total_colunas": dados_leads_enriquecidos.shape[1],
    "total_outliers_score": len(outliers_score),
    "percentual_outliers_score": round(percentual_outliers, 2),
    "total_leads_criticos": len(leads_criticos),
    "percentual_leads_criticos": round(
        len(leads_criticos) / len(dados_leads_enriquecidos) * 100,
        2
    )
}

resumo_engenharia

{'total_leads': 950000,
 'total_colunas': 14,
 'total_outliers_score': 12263,
 'percentual_outliers_score': 1.29,
 'total_leads_criticos': 341855,
 'percentual_leads_criticos': 35.98}

In [31]:
#Salvar amostra enriquecida

CAMINHO_OUTPUTS = RAIZ_PROJETO / "outputs"
CAMINHO_OUTPUTS.mkdir(parents=True, exist_ok=True)

CAMINHO_AMOSTRA_ENRIQUECIDA = CAMINHO_OUTPUTS / "amostra_leads_enriquecida.csv"

dados_leads_enriquecidos.sample(
    n=min(10_000, len(dados_leads_enriquecidos)),
    random_state=42
).to_csv(
    CAMINHO_AMOSTRA_ENRIQUECIDA,
    index=False,
    encoding="utf-8-sig"
)

print("Amostra enriquecida salva em:", CAMINHO_AMOSTRA_ENRIQUECIDA)

Amostra enriquecida salva em: c:\Users\DataCore\Desktop\projetos_Git\agro-leads-orchestrator\outputs\amostra_leads_enriquecida.csv


In [32]:
#Salvar resumos de nulos

CAMINHO_RESUMO_NULOS = CAMINHO_OUTPUTS / "resumo_nulos.csv"

resumo_nulos.to_csv(
    CAMINHO_RESUMO_NULOS,
    index=False,
    encoding="utf-8-sig"
)

print("Resumo de nulos salvo em:", CAMINHO_RESUMO_NULOS)

Resumo de nulos salvo em: c:\Users\DataCore\Desktop\projetos_Git\agro-leads-orchestrator\outputs\resumo_nulos.csv


In [33]:
conexao.close()

print("Conexão encerrada.")

Conexão encerrada.
